# HOGENOM CCP Adam optimization

Specieswise optimization of the HOGENOM CCP likelihood with Adam, an oscillation-based learning-rate decay, and a Beta prior on `p_S`. The prior below uses `Beta(4, 1)`, which puts about 99.84% prior mass above `p_S = 0.2`.

In [ ]:
from __future__ import annotations

import json
import math
import sys
import time
from pathlib import Path

import torch


def find_repo(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "gpurec").is_dir() and (path / "tests").is_dir():
            return path
    raise RuntimeError("could not find gpurec repo root")


REPO = find_repo(Path.cwd().resolve())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from gpurec import GeneReconModel

HOGENOM_DIR = REPO / "tests" / "data" / "HOGENOM" / "hogenom"
FAMILIES_FILE = HOGENOM_DIR / "hogenom_families.local.txt"
ALERAX_OUTPUT = HOGENOM_DIR / "output_alerax_corrected"
INFERRED_SPECIES_TREE = ALERAX_OUTPUT / "species_trees" / "inferred_species_tree.newick"
SPECIES_TREE = INFERRED_SPECIES_TREE if INFERRED_SPECIES_TREE.exists() else HOGENOM_DIR / "hogenom_S.tree"
PREPROCESS_CACHE = HOGENOM_DIR / "output_gpurec_ccp_reconciliation" / "preprocess_cache"
OUT_DIR = HOGENOM_DIR / "output_gpurec_adam_oscillation_notebook"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("This notebook expects the CUDA fast path.")

# Dataset/model knobs. Set MAX_FAMILIES to a small number for a quick smoke run.
MAX_FAMILIES = None
FAMILY_CHUNK_SIZE = 0
CLADE_BUDGET = 305_000
MAX_WAVE_SIZE = 8192
FIXED_ITERS_E = 6
FIXED_ITERS_PI = 6
NEUMANN_TERMS = 6

# Adam and safety bounds.
STEPS = 5000
LR = 0.01
MIN_RATE = 1e-10
MAX_RATE = 100.0

# Oscillation detector. A negative cosine catches direction reversals;
# the sign-flip fraction catches coordinate-wise bouncing.
OSCILLATION_DETECTION = "both"  # "gradient", "parameters", "both", or "off"
OSCILLATION_COS_THRESHOLD = -0.25
OSCILLATION_FLIP_FRACTION = 0.5
OSCILLATION_LR_DECAY = 0.5
OSCILLATION_COOLDOWN = 5
MIN_LR = 1e-6

# Beta prior on p_S. Beta(4, 1) strongly favors p_S > 0.2.
BETA_PS_ALPHA = 4.0
BETA_PS_BETA = 1.0
BETA_PRIOR_WEIGHT = 1.0

PRINT_EVERY = 1
HISTORY_PATH = OUT_DIR / "history.jsonl"
THETA_PATH = OUT_DIR / "theta_final.pt"

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("repo", REPO)
print("species_tree", SPECIES_TREE)
print("families_file", FAMILIES_FILE)
print("out_dir", OUT_DIR)

In [ ]:
LN2 = math.log(2.0)
RATE_QUANTILES = torch.tensor([0.0, 0.05, 0.5, 0.95, 1.0])


def sync() -> None:
    torch.cuda.synchronize()


def theta_logits(theta: torch.Tensor) -> torch.Tensor:
    theta2 = theta.reshape(-1, 3)
    zeros = theta2.new_zeros((theta2.shape[0], 1))
    return torch.cat((zeros, theta2), dim=1) * LN2


def pS_values(theta: torch.Tensor) -> torch.Tensor:
    return torch.softmax(theta_logits(theta), dim=1)[:, 0]


def beta_ps_prior_bits(theta: torch.Tensor) -> torch.Tensor:
    logits = theta_logits(theta)
    log_probs_bits = torch.log_softmax(logits, dim=1) / LN2
    log_pS = log_probs_bits[:, 0]
    log_not_pS = (
        torch.logsumexp(logits[:, 1:], dim=1) / LN2
        - torch.logsumexp(logits, dim=1) / LN2
    )
    penalty = -(
        (BETA_PS_ALPHA - 1.0) * log_pS
        + (BETA_PS_BETA - 1.0) * log_not_pS
    ).sum()
    return BETA_PRIOR_WEIGHT * penalty


def quantile_summary(values: torch.Tensor) -> dict[str, float]:
    q = torch.quantile(values.detach().float().cpu(), RATE_QUANTILES)
    return {
        "min": float(q[0]),
        "p05": float(q[1]),
        "median": float(q[2]),
        "p95": float(q[3]),
        "max": float(q[4]),
    }


def rate_summary(theta: torch.Tensor) -> dict[str, dict[str, float]]:
    theta2 = theta.detach().reshape(-1, 3)
    rates = torch.exp2(theta2)
    return {
        "D": quantile_summary(rates[:, 0]),
        "T": quantile_summary(rates[:, 2]),
        "L": quantile_summary(rates[:, 1]),
        "pS": quantile_summary(pS_values(theta2)),
    }


def format_summary(summary: dict[str, dict[str, float]]) -> str:
    return " ".join(
        f"{k}[min={v['min']:.3g} med={v['median']:.3g} p95={v['p95']:.3g} max={v['max']:.3g}]"
        for k, v in summary.items()
    )


def tensor_is_finite(x: torch.Tensor) -> bool:
    return bool(torch.isfinite(x).all().item())


def tensor_cosine(previous: torch.Tensor | None, current: torch.Tensor | None) -> float | None:
    if previous is None or current is None:
        return None
    if not tensor_is_finite(previous) or not tensor_is_finite(current):
        return None
    a = previous.detach().reshape(-1)
    b = current.detach().reshape(-1)
    denom = torch.linalg.vector_norm(a) * torch.linalg.vector_norm(b)
    if float(denom.cpu()) == 0.0:
        return None
    return float(torch.dot(a, b).div(denom).cpu())


def sign_flip_fraction(previous: torch.Tensor | None, current: torch.Tensor | None) -> float | None:
    if previous is None or current is None:
        return None
    if not tensor_is_finite(previous) or not tensor_is_finite(current):
        return None
    active = (previous != 0) & (current != 0)
    n = int(active.sum().cpu())
    if n == 0:
        return None
    flips = (previous.sign() != current.sign()) & active
    return float(flips.sum().cpu()) / n


def maybe_decay_adam_lr(optimizer, previous_grad, grad, previous_step, step, cooldown):
    if OSCILLATION_DETECTION == "off":
        return cooldown, {"adam_lr": optimizer.param_groups[0]["lr"]}

    reasons = []
    metrics = {"adam_lr": optimizer.param_groups[0]["lr"], "adam_lr_reduced": False}

    if OSCILLATION_DETECTION in ("gradient", "both"):
        grad_cos = tensor_cosine(previous_grad, grad)
        grad_flips = sign_flip_fraction(previous_grad, grad)
        metrics.update(grad_cosine=grad_cos, grad_flip_fraction=grad_flips)
        if grad_cos is not None and grad_cos <= OSCILLATION_COS_THRESHOLD:
            reasons.append(f"grad_cos={grad_cos:.3g}")
        if grad_flips is not None and grad_flips >= OSCILLATION_FLIP_FRACTION:
            reasons.append(f"grad_flips={grad_flips:.3g}")

    if OSCILLATION_DETECTION in ("parameters", "both"):
        step_cos = tensor_cosine(previous_step, step)
        step_flips = sign_flip_fraction(previous_step, step)
        metrics.update(step_cosine=step_cos, step_flip_fraction=step_flips)
        if step_cos is not None and step_cos <= OSCILLATION_COS_THRESHOLD:
            reasons.append(f"step_cos={step_cos:.3g}")
        if step_flips is not None and step_flips >= OSCILLATION_FLIP_FRACTION:
            reasons.append(f"step_flips={step_flips:.3g}")

    if cooldown > 0:
        metrics["oscillation_reason"] = ",".join(reasons) if reasons else ""
        return cooldown - 1, metrics
    if not reasons:
        return cooldown, metrics

    old_lr = optimizer.param_groups[0]["lr"]
    new_lr = max(MIN_LR, old_lr * OSCILLATION_LR_DECAY)
    if new_lr < old_lr:
        for group in optimizer.param_groups:
            group["lr"] = new_lr
        metrics.update(
            adam_lr=new_lr,
            adam_lr_previous=old_lr,
            adam_lr_reduced=True,
            oscillation_reason=",".join(reasons),
        )
        cooldown = OSCILLATION_COOLDOWN
    return cooldown, metrics

In [ ]:
build_start = time.perf_counter()
model = GeneReconModel.from_alerax_families(
    str(SPECIES_TREE),
    FAMILIES_FILE,
    mode="specieswise",
    start=0,
    max_families=MAX_FAMILIES,
    device=DEVICE,
    dtype=torch.float32,
    theta_init_rates=(0.05, 0.05, 0.05),
    preprocess_cache_dir=PREPROCESS_CACHE,
    fixed_iters_E=FIXED_ITERS_E,
    fixed_iters_Pi=FIXED_ITERS_PI,
    neumann_terms=NEUMANN_TERMS,
    family_chunk_size=FAMILY_CHUNK_SIZE,
    clade_budget=CLADE_BUDGET,
    batch_packing="depth_first_fit",
    max_wave_size=MAX_WAVE_SIZE,
    lazy_preprocess=True,
    prefetch_batches="all",
)

for batch_idx in range(len(model.batch_metadata)):
    model._ensure_batch_static(batch_idx)
model.clear()
sync()

print(
    f"build_s={time.perf_counter() - build_start:.3f} "
    f"families={sum(m.family_count for m in model.batch_metadata)} "
    f"species={model.n_species} batches={len(model.batch_metadata)} "
    f"waves={sum(m.wave_count for m in model.batch_metadata)}"
)

In [ ]:
optimizer = torch.optim.Adam([model.theta], lr=LR)
history: list[dict[str, float | int | str | bool | None]] = []
if HISTORY_PATH.exists():
    HISTORY_PATH.unlink()

previous_objective = None
previous_grad = None
previous_step = None
cooldown = 0

for iteration in range(STEPS):
    step_start = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)
    model.theta.grad = None

    eval_start = time.perf_counter()
    data_nll = model.full_loss()
    prior = beta_ps_prior_bits(model.theta)
    objective = data_nll + prior
    objective.backward()
    sync()
    eval_s = time.perf_counter() - eval_start

    if model.theta.grad is None:
        raise RuntimeError("missing theta gradient")
    if not tensor_is_finite(objective) or not tensor_is_finite(model.theta.grad):
        print("stopping: non-finite objective or gradient")
        break

    theta_before = model.theta.detach().clone()
    grad = model.theta.grad.detach().clone()
    optimizer.step()
    sync()
    model.clamp_theta_(min_rate=MIN_RATE, max_rate=MAX_RATE)
    step = (model.theta.detach() - theta_before).detach().clone()
    cooldown, adam_metrics = maybe_decay_adam_lr(
        optimizer, previous_grad, grad, previous_step, step, cooldown
    )

    objective_bits = float(objective.detach().cpu())
    data_nll_bits = float(data_nll.detach().cpu())
    prior_bits = float(prior.detach().cpu())
    delta = None if previous_objective is None else previous_objective - objective_bits
    previous_objective = objective_bits

    summary = rate_summary(model.theta)
    row = {
        "iteration": iteration,
        "objective_bits": objective_bits,
        "data_nll_bits": data_nll_bits,
        "beta_ps_prior_bits": prior_bits,
        "delta_objective_bits": delta,
        "grad_inf": float(grad.abs().amax().cpu()),
        "grad_norm": float(torch.linalg.vector_norm(grad).cpu()),
        "theta_step_inf": float(step.abs().amax().cpu()),
        "eval_s": eval_s,
        "step_s": time.perf_counter() - step_start,
        "rates": summary,
        **adam_metrics,
    }
    history.append(row)
    with HISTORY_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, sort_keys=True) + "\n")

    previous_grad = grad
    previous_step = step

    if iteration % PRINT_EVERY == 0:
        delta_text = "nan" if delta is None else f"{delta:.6g}"
        lr_text = f"lr={optimizer.param_groups[0]['lr']:.3g}"
        decay_text = ""
        if adam_metrics.get("adam_lr_reduced"):
            decay_text = f" lr_reduced reason={adam_metrics.get('oscillation_reason', '')}"
        print(
            f"iter={iteration:04d} objective_bits={objective_bits:.6f} "
            f"data_nll_bits={data_nll_bits:.6f} prior_bits={prior_bits:.3f} "
            f"delta={delta_text} grad_inf={row['grad_inf']:.6g} "
            f"grad_norm={row['grad_norm']:.6g} theta_step_inf={row['theta_step_inf']:.3g} "
            f"eval_s={eval_s:.3f} step_s={row['step_s']:.3f} {lr_text}{decay_text}"
        )
        print("  " + format_summary(summary))

torch.save(model.theta.detach().cpu(), THETA_PATH)
print("saved", THETA_PATH)
print("history", HISTORY_PATH)

In [ ]:
# Exact final evaluation at the saved theta.
optimizer.zero_grad(set_to_none=True)
model.theta.grad = None
start = time.perf_counter()
data_nll = model.full_loss()
prior = beta_ps_prior_bits(model.theta)
objective = data_nll + prior
objective.backward()
sync()
print(
    f"final objective_bits={float(objective.detach().cpu()):.6f} "
    f"data_nll_bits={float(data_nll.detach().cpu()):.6f} "
    f"prior_bits={float(prior.detach().cpu()):.3f} "
    f"grad_inf={float(model.theta.grad.detach().abs().amax().cpu()):.6g} "
    f"eval_s={time.perf_counter() - start:.3f}"
)
print(format_summary(rate_summary(model.theta)))